<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

### RapidFire AI RAG/Context Engineering Tutorial Use Case: RapidFire Documentation Q&A


In [ ]:
from pathlib import Path

TRITON_BASE_URL = "https://tritonai-api.ucsd.edu/v1"
TRITON_API_KEY_PATH = next(
    path for path in [Path("~/api.txt").expanduser(), Path("~/api-key.txt").expanduser()]
    if path.exists()
)
TRITON_API_KEY = TRITON_API_KEY_PATH.read_text(encoding="utf-8").splitlines()[0].strip()


In [ ]:
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
from rapidfireai import Experiment

import json
import math
import re
from functools import lru_cache
from pathlib import Path
from typing import List as listtype, Dict, Any

import pandas as pd
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


##### API Cost Considerations
This notebook runs several small validation-set configs over 45 documentation questions.

The TritonAI gateway is used for generation. Retrieval tuning is local and does not call the API.


### Load Dataset and Rename Columns

In [ ]:
DATA_ROOT = next(path for path in [Path("data"), Path("../../data")] if path.exists())
SOURCE_DOCS_DIR = DATA_ROOT / "sourcedocs"
VALIDATION_PATH = DATA_ROOT / "validation" / "validation-set-golden-qa-pairs.json"

with open(VALIDATION_PATH, "r", encoding="utf-8") as f:
    validation_examples = json.load(f)

rag_dataset = Dataset.from_dict(
    {
        "query": [example["question"] for example in validation_examples],
        "query_id": [int(example["question_id"]) for example in validation_examples],
        "reference_answer": [example["reference_answer"] for example in validation_examples],
        "expected_source_files": [
            sorted({evidence["file"] for evidence in example.get("source_evidence", [])})
            for example in validation_examples
        ],
    }
)

pd.DataFrame(rag_dataset).head()


### Create Experiment

In [ ]:
experiment = Experiment(experiment_name="discussion1-rag-docs-tuning-8configs", mode="evals")


### Define Local Retrieval Configs


In [ ]:
batch_size = 16

from discussion_retrieval_utils import (
    RETRIEVAL_CONFIGS,
    retrieve_contexts,
    preprocess_word_192_16,
    preprocess_word_128_32,
    preprocess_char_128_96,
    preprocess_word_96_64,
    preprocess_word_96_32,
    preprocess_char_96_64,
    preprocess_word_192_64,
    sample_postprocess_fn,
    sample_compute_metrics_fn,
    sample_accumulate_metrics_fn,
)

RETRIEVAL_CONFIGS


### Define Data Processing and Postprocessing Functions

In [ ]:
from discussion_retrieval_utils import INSTRUCTIONS
print(INSTRUCTIONS)


In [ ]:
# Preprocess and postprocess functions are imported from discussion_retrieval_utils.py.
# Keeping them in a module makes RapidFire/Ray serialization reliable.


### Define Custom Eval Metrics Functions

In [ ]:
# Metric functions are imported from discussion_retrieval_utils.py.


### Define TritonGPT Generator and Retrieval Configs


In [ ]:
triton_gpt_config_1024 = RFOpenAIAPIModelConfig(
    client_config={"api_key": TRITON_API_KEY, "base_url": TRITON_BASE_URL, "max_retries": 2},
    model_config={
        "model": "api-gpt-oss-120b",
        "max_completion_tokens": 1024,
    },
    rpm_limit=120,
    tpm_limit=1_000_000,
    rag=None,
    prompt_manager=None,
)

triton_gpt_config_1536 = RFOpenAIAPIModelConfig(
    client_config={"api_key": TRITON_API_KEY, "base_url": TRITON_BASE_URL, "max_retries": 2},
    model_config={
        "model": "api-gpt-oss-120b",
        "max_completion_tokens": 1536,
    },
    rpm_limit=120,
    tpm_limit=1_000_000,
    rag=None,
    prompt_manager=None,
)

base_eval_kwargs = {
    "batch_size": batch_size,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

config_set = List(
    [
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_word_192_16},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_word_128_32},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_char_128_96},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1536, "preprocess_fn": preprocess_word_192_16},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_word_96_64},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_word_96_32},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_char_96_64},
        {**base_eval_kwargs, "openai_config": triton_gpt_config_1024, "preprocess_fn": preprocess_word_192_64},
    ]
)


### Create Config Group

In [ ]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

### Run Multi-Config Evals

In [ ]:
# Launch evals of all configs in the config_group.
results = experiment.run_evals(
    config_group=config_group,
    dataset=rag_dataset,
    num_actors=2,
    num_shards=4,
    seed=42,
)


### View Results

In [ ]:
# Convert results dict to DataFrame
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

results_df.sort_values("Total Score", ascending=False)


### End Experiment

In [ ]:
experiment.end()

### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")

In [ ]:
import json, pandas as pd

best_summary_path = "rag_outputs/output_char_128_96_reference_like_1024.summary.json"
with open(best_summary_path, "r", encoding="utf-8") as f:
    best_summary = json.load(f)

pd.DataFrame([best_summary])[[
    "output", "judge_report", "retrieval_score",
    "partial_generation_score", "partial_total_score"
]]
